Linear regression takes a set of 1D input data and 1D output data

We are looking for a line of best fit for the data

We can do it in 2 ways, using a system of 2 equations or linear algebra

let's start by writing down out matrix

In [219]:
import numpy as np

In [178]:
x1 = np.array([5,4,6,6,7])
x2 = np.array([2.3, 2.4, 3.0,2.4,2.6])

Start with linear algebra

Now we want to create a matrix and it's transpose from the data

In [179]:
x0 = np.ones(5)
X_T = np.vstack((x0,x1))

In [180]:
X = X_T.transpose()

In [181]:
y = x2.transpose()

In [182]:
X_T_X = X_T @ X
X_T_y = X_T @ y

X_T_X_inv = np.linalg.inv(X_T_X)

beta = X_T_X_inv @ X_T_y

In [183]:
beta

array([1.91538462, 0.11153846])

Ok let's try it via formulas and see if we get the same answer

We're gonna need to calculate sum of x, y, xy, and x^2, and the means will be sums divided by n

In [184]:
n = len(x1)

sum_x = np.sum(x1)
sum_y = np.sum(x2)
sum_xy = np.sum(x1 * x2)
sum_xx = np.sum(x1 * x1)

B_1 = ( n * sum_xy - sum_x * sum_y ) / (n * sum_xx - sum_x * sum_x )
B_0 = ( sum_y - B_1 * sum_x ) / n


In [185]:
B_0

np.float64(1.915384615384608)

BOOM

What about L2 or Ridge Regression??

Well it's the same but now with a weight on the slope, basically assuming we don't have enough data and we want to add some bias. That is quite relevant for our case here

There is just a few formula we want to optimize

Before it was sum((B_0 + B_1x - y)^2). But now it's sum( (b_0 + B_1x - y)^2 ) + b_1 ^ 2

In [186]:
def dB_0(beta, x, y):
    return 2 * (beta[0] + beta[1] * x - y)

In [187]:
def dB_1(beta, x, y, lam):
    return 2 * x * (beta[0] + beta[1] * x - y) + 2 * lam * beta[1]

First since I'm not sure how to do this analytically let's solve this with gradient descent

In [210]:
x1 = np.array([5,4,6,6,7])
x2 = np.array([2.3, 2.4, 3.0,2.4,2.6])
n = len(x1)

Best guess for beta and set parameters

In [216]:
beta = np.array([0.0,0.0])
maxiter = 10000
acceptance = 0.01
learning_rate = 0.001
converged = False
lam = 0

In [217]:
for i in range(maxiter):
    dB_0_grad = 0
    dB_1_grad = 0
    for j in range(len(x1)):
        dB_0_grad += dB_0(beta, x1[j], x2[j])
        dB_1_grad += dB_1(beta, x1[j], x2[j], lam)
    dB_0_grad /= n
    dB_1_grad /= n

    if np.sqrt(dB_0_grad ** 2 + dB_1_grad ** 2 ) < acceptance:
        print(f"Converged at iteration {i}")
        converged = True
        break
    else:
        beta = beta - learning_rate * np.array([dB_0_grad, dB_1_grad])


In [218]:
beta

array([0.9283545 , 0.28230052])

Now let's do it analytically using the following equations

B_0 = y hat - B_1 * x hat

the same as normal

B_1 = sum ( xi - x hat ) (yi - y hat) / ( sum ( xi - x hat ) ^ 2 + lam )

In [192]:
x_hat = np.sum(x1) / n
y_hat = np.sum(x2) / n

In [197]:
B_1 = np.sum((x1 - x_hat) * (x2 - y_hat)) / (np.sum((x1 - x_hat) ** 2)  + lam)
B_0 = y_hat - B_1 * x_hat

In [198]:
B_0

np.float64(1.9153846153846148)

In [199]:
B_1

np.float64(0.11153846153846163)

Actually it seems like we can do the exact same thing for multiple regression if we just start with more data

In [232]:
x1 = np.array([5,4,6,6,7])
x2 = np.array([2.3, 2.4, 3.0,2.4,2.6])
x3 = np.array([0.19, 0.2, 0.22, 0.19, 0.23])
x0 = np.ones(5)

In [233]:
X = np.column_stack((x0,x1,x2))
X_T = np.transpose(X)

Y = np.transpose(x3)

In [234]:
np.linalg.inv(X_T @ X) @ (X_T @ Y)

array([0.08029549, 0.0059409 , 0.03639191])

Ah and this leads to how to do polynomial regression

In [235]:
x1 = np.array([5,4,6,6,7])
y = np.array([2.3, 2.4, 3.0,2.4,2.6])
x2 = x1 ** 2
x0 = np.ones(5)

In [236]:
X = np.column_stack((x0,x1,x2))
X_T = np.transpose(X)

Y = np.transpose(y)

np.linalg.inv(X_T @ X) @ (X_T @ Y)

array([ 1.21935484,  0.37580645, -0.02419355])

combining multiple regression with quadratic polynomials

That is, 2 dimensional input (multiple regression)

least sqaures regression (the formula we've been using!)

Quadratic polynomial

In [237]:
x1 = np.array([5,4,6,6,7])
x2 = np.array([2.3, 2.4, 3.0,2.4,2.6])
x1_x2 = x1 * x2
x1_srd = x1 ** 2
x2_srd = x2 ** 2
y = np.array([0.19, 0.2, 0.22, 0.19, 0.23])
x0 = np.ones(5)

In [239]:
X = np.column_stack((x0, x1,x2,x1_x2,x1_srd,x2_srd))
Y = np.transpose(y)
X_T = np.transpose(X)

np.linalg.inv(X_T @ X) @ (X_T @ Y)

array([ 1.671875  , -0.2421875 , -0.65625   ,  0.05761719,  0.01000977,
        0.0703125 ])